# Reproduce the paper’s tables and figures

**Run all cells from top to bottom.** The notebook calls functions in `codes/` and saves only the paper’s 12 tables, four item examples, and six figure files in `output/`. The analysis clears previous files in `output/` when it starts.

## Setup
Use Python 3.12 on macOS or Linux and install the pinned dependencies in the environment selected as the notebook kernel:
```
python -m pip install -r codes/requirements.txt
```
No GPU or new model inference is required.

## Starting data
The notebook first checks `rawdata/`. If the required inputs are already present, it uses them without changing them. If the folder is missing, empty, or incomplete, it downloads the evaluation logs and metadata from Hugging Face, regrades MATH answers using Math-Verify, and builds the input tables in `rawdata/` before starting the analysis.

Downloading requires a Hugging Face account authorized to access the evaluation-log datasets. Your saved local login is used automatically. If no token is saved, the notebook asks for it through a hidden prompt; never paste a token into a code cell. Download progress is resumable, and temporary source records are removed after a successful full build. A fresh download and regrading take longer than running the analyses from existing inputs.

The source manifest fixes the model list, evaluation dates and canonical question-version identifiers. The complete fresh-data rebuild has not yet been verified end to end; the tested models matched the supplied correctness data. The final notebook cell checks the main numerical results and the set of paper outputs.

The statistical procedures and seeds are explained alongside each step. Intermediate analysis estimates and predictions stay in memory. The analysis-only notebook was previously validated on an Apple M1 host (8 logical CPUs, 16 GiB RAM), using one BLAS thread. The last cell reports analysis runtime and peak process memory.

In [ ]:
import os, sys, warnings
from pathlib import Path
sys.dont_write_bytecode = True
# Expected only for MH questions with no finite estimate; their counts are reported.
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice.|All-NaN slice encountered', category=RuntimeWarning)
for variable in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[variable] = '1'
root = Path.cwd()
if not (root / 'codes').is_dir():
    raise RuntimeError('Open this notebook from the reproducibility folder.')
sys.path.insert(0, str(root))
from IPython.display import display
from codes import pipeline, primary, content, cross_validation
from codes.examples import item_examples

## Prepare the input data
Use the existing `rawdata/`, or download and rebuild missing inputs there. A complete existing dataset does not require a Hugging Face login.

In [ ]:
from codes import data

data.ensure_inputs()
study = pipeline.start()

## 1. Model selection and metadata
Apply the paper's family, six-benchmark coverage, identical-correctness deduplication and completeness filters. Write the model-selection table and display the configuration/uploader summaries used in the appendix.

In [ ]:
display(primary.selection(study))
display(study.results['model_metadata'])
display(study.results['lineage_summary'])

## 2. Primary MH DIF: Table 1
Retain questions with pooled correctness from 1% through 99%. Scores are equal-item mean accuracy. Both families use shared absolute-score intervals, merged until each contains at least five models per family, with a five-percentage-point maximum width. MH inference uses 1,000 linearized resamples of recorded-lineage clusters (seed 20261030); BH corrections for zero-effect and beyond-one-delta tests are separate within each benchmark. The original A/B/C rules and non-finite-estimate reporting convention are preserved.

In [ ]:
display(primary.prevalence(study))

## 3. Matching-score sensitivity and appendix robustness tables
Recalculate scores once using initial Category-A questions. Also evaluate the fixed 1-, 2-, 3-, and 5-point intervals shown in the paper. Keep the question sample fixed within these sensitivity comparisons. Write only the three published sensitivity/coverage tables.

In [ ]:
anchors, intervals = primary.sensitivities(study)
display(anchors)
display(intervals)

## 4. Additional model-family comparisons
Repeat the same procedure separately for Qwen–Mistral and Llama–Mistral. Completeness, deduplication and 1–99% filtering are pair-specific. Mistral is the reference family in both tables. Coverage and non-finite estimates are displayed alongside prevalence to make the supported scope explicit.

In [ ]:
display(primary.additional_families(study))

## 5. Nonuniform DIF
For each retained question, regress correctness on matching score, family, and their interaction, using models in supported primary strata. Test the interaction using lineage-cluster standard errors, then apply BH within each benchmark. The fitting routine checks convergence and numerical testability; untestable fits are not treated as evidence of no DIF. Percentages use all questions, or all questions in the specified MH category, as their denominator. Write the nonuniform-DIF table.

The paper export still has older interaction percentages in the preceding robustness prose. The updated table and the following displayed score-dependence summaries use the corrected fitting routine.

In [ ]:
display(primary.nonuniform(study))
display(study.results['score_dependence'])

## 6. DIF across benchmark subcategories
Run the two Pearson chi-square tests using subcategories with at least 30 retained questions: subcategory × A/B/C and subcategory × direction among C questions. IFEval uses the first instruction's group. Write the main categorical table and the two category figures. Calculate the GPQA subdomain tests and display the detailed breakdowns underlying the appendix prose.

In [ ]:
display(content.categories(study))
display(study.results['category_details'])
display(study.results['gpqa_subdomains'])

## 7. Four individual examples
Select the same four BBH/MuSR Category-C examples using the documented task, correct-answer, text-length and effect-size rules. The displayed median stratum contains the median-score model among matched models. Write only the example LaTeX block used in the paper.

In [ ]:
display(item_examples(study))

## 8. Correct-answer position and DIF direction
Among Category-C questions, fit Firth logistic regressions separately by benchmark and number of options, with additive subcategory and answer-position predictors. Wald inference and separate BH corrections cover all 72 position contrasts and 13 joint tests. Write only the 13-row summary table; keep full coefficient estimates in memory.

In [ ]:
display(content.position_regressions(study))

## 9. Actual BBH answer selections
Use all retained questions in the eight binary tasks. Give each correct answer equal weight when calculating each model's selection rate. Adjust for overall BBH accuracy and refit the regression in 1,000 resamples of whole lineage clusters (seed 20261106). Write the selection table, task-by-correct-answer table, and the single PDF figure. The unused MuSR selection analysis is not run.

In [ ]:
display(content.answer_selections(study))

## 10. Predicted sets and held-out performance
Use five folds stratified by subcategory (seed 20261105), separately for BBH and MuSR. Recalculate scores, strata and MH labels on each 80% training split. Fit the additive one-hot subcategory + option-count:position classifier on training Category-C items, using logistic regression with L2 regularization, C=1, no class weights, and at most 2,000 iterations. Predict Qwen/Llama for every held-out question.

Compare held-out accuracy within each training-defined supported stratum, weighting strata by pooled model counts. Bootstrap questions 1,000 times (seed 20261109), preserving each question's fold and assignment; models, rules and strata stay fixed. The same sampled questions are used for both families and all sets. Write the main overall plot and the two fold-by-stratum PDFs. There is no model bootstrap, random-subset comparator or Bayesian bootstrap here.

Features were explored using the full datasets, and eligibility is fixed before splitting. This is held-out evaluation of the fitted rule, not independent confirmation of the feature hypothesis.

In [ ]:
display(cross_validation.run(study))

## 11. Check results and final outputs
Check the main numerical results and require the output directory to contain exactly the paper's 19 artifacts, with no CSV files, cached predictions or duplicate figure formats. Report actual elapsed time and peak process memory for this run. The assertions below compare computed results with published rounded values; they are checks, not analysis inputs.

In [ ]:
display(pipeline.validate(study))